# Data Preparation for GWM-RNN Relation Prediction - Wikidata5M

This notebook prepares the Wikidata5M dataset for training GWM-RNN on the Knowledge Graph Completion task.

## Dataset Overview
- **Wikidata5M**: Large-scale subset of Wikidata knowledge graph
- **Entities**: ~4.8M entities (various real-world entities)
- **Relations**: 822 relation types
- **Triples**: ~20.6M training, ~5K validation, ~5K test
- **Task**: Given `(head, relation, ?)`, predict the tail entity
- **Source**: https://deepgraphlearning.github.io/project/wikidata5m

## Dataset Files
- `wikidata5m_transductive_train.txt`: Training triples
- `wikidata5m_transductive_valid.txt`: Validation triples
- `wikidata5m_transductive_test.txt`: Test triples
- `wikidata5m_entity.txt`: Entity descriptions
- `wikidata5m_relation.txt`: Relation names and aliases

## Processing Steps
1. Load raw triples from text files
2. Create entity and relation vocabularies
3. Generate inverse relations (doubles the data)
4. Encode entity/relation descriptions using Sentence-BERT
5. Save processed tensors for training
6. Generate context embeddings from training graph only (NO DATA LEAKAGE)

## 1. Setup and Configuration

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict
import json

# Paths
RAW_DATA_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\data\wikidata5m\raw")
OUTPUT_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\data\wikidata5m\processed\relation-prediction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'  # 384-dim embeddings
BATCH_SIZE = 512  # Larger batch for efficiency with large dataset
DEVICE = 'cuda'

# Inverse relation settings
CREATE_INVERSE_RELATIONS = False  # Double the data by adding inverse triples

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Create inverse relations: {CREATE_INVERSE_RELATIONS}")
print(f"\n⚠️  Note: Wikidata5M is a LARGE dataset (~4.8M entities, ~20M triples)")
print(f"   Processing may take significant time and memory")

## 2. Load Raw Data

In [ ]:
def load_triples(file_path):
    """Load triples from tab-separated file."""
    triples = []
    print(f"Loading triples from {file_path}...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Reading triples"):
            parts = line.strip().split('\t')
            if len(parts) == 3:
                h, r, t = parts
                triples.append((h, r, t))
    return triples

# Load all splits
train_triples = load_triples(RAW_DATA_DIR / 'wikidata5m_transductive_train.txt')
valid_triples = load_triples(RAW_DATA_DIR / 'wikidata5m_transductive_valid.txt')
test_triples = load_triples(RAW_DATA_DIR / 'wikidata5m_transductive_test.txt')

print(f"\nTraining triples: {len(train_triples):,}")
print(f"Validation triples: {len(valid_triples):,}")
print(f"Test triples: {len(test_triples):,}")
print(f"Total triples: {len(train_triples) + len(valid_triples) + len(test_triples):,}")

# Show examples
print("\nExample triples:")
for i, triple in enumerate(train_triples[:5], 1):
    print(f"{i}. {triple}")

## 3. Create Vocabularies and Inverse Relations

In [ ]:
def create_vocabularies(train_triples, valid_triples, test_triples, create_inverse=True):
    """
    Create entity and relation vocabularies.
    Optionally add inverse relations.
    """
    entities = set()
    relations = set()
    
    # Collect all entities and relations
    all_triples = train_triples + valid_triples + test_triples
    print("Collecting entities and relations...")
    for h, r, t in tqdm(all_triples, desc="Processing triples"):
        entities.add(h)
        entities.add(t)
        relations.add(r)
    
    # Add inverse relations
    if create_inverse:
        original_relations = list(relations)
        for rel in original_relations:
            relations.add(rel + '_inv')
    
    # Create mappings
    print("Creating vocabularies...")
    entity2id = {ent: idx for idx, ent in enumerate(sorted(entities))}
    id2entity = {idx: ent for ent, idx in entity2id.items()}
    
    relation2id = {rel: idx for idx, rel in enumerate(sorted(relations))}
    id2relation = {idx: rel for rel, idx in relation2id.items()}
    
    return entity2id, id2entity, relation2id, id2relation

entity2id, id2entity, relation2id, id2relation = create_vocabularies(
    train_triples, valid_triples, test_triples, 
    create_inverse=CREATE_INVERSE_RELATIONS
)

print(f"\nNumber of entities: {len(entity2id):,}")
print(f"Number of relations: {len(relation2id):,}")

if CREATE_INVERSE_RELATIONS:
    original_rels = [r for r in relation2id.keys() if not r.endswith('_inv')]
    inverse_rels = [r for r in relation2id.keys() if r.endswith('_inv')]
    print(f"  - Original relations: {len(original_rels)}")
    print(f"  - Inverse relations: {len(inverse_rels)}")

# Save vocabularies
print("\nSaving vocabularies...")
with open(OUTPUT_DIR / 'entity2id.json', 'w') as f:
    json.dump(entity2id, f, indent=2)
with open(OUTPUT_DIR / 'relation2id.json', 'w') as f:
    json.dump(relation2id, f, indent=2)

print("✓ Vocabularies saved")

## 4. Load Entity and Relation Descriptions

In [ ]:
def load_entity_descriptions(file_path):
    """
    Load entity descriptions from wikidata5m_entity.txt.
    Format: entity_id<TAB>description
    """
    descriptions = {}
    print(f"Loading entity descriptions from {file_path}...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Reading entity descriptions"):
            parts = line.strip().split('\t', 1)  # Split on first tab only
            if len(parts) == 2:
                entity_id, description = parts
                descriptions[entity_id] = description.strip()
            elif len(parts) == 1:
                # Entity with no description
                entity_id = parts[0]
                descriptions[entity_id] = entity_id
    return descriptions

# Load entity descriptions
entity_descriptions = load_entity_descriptions(RAW_DATA_DIR / 'wikidata5m_entity.txt')
print(f"\nLoaded {len(entity_descriptions):,} entity descriptions")

# Create entity texts (with fallback to entity ID)
entity_texts = {}
print("\nCreating entity texts...")
for entity in tqdm(entity2id.keys(), desc="Processing entities"):
    if entity in entity_descriptions:
        text = entity_descriptions[entity]
        # Truncate very long descriptions to avoid memory issues
        if len(text) > 500:
            text = text[:497] + '...'
        entity_texts[entity] = text
    else:
        # Clean up entity ID for readability
        clean_id = entity.replace('_', ' ').replace('/', ' ').strip()
        entity_texts[entity] = clean_id

print(f"Entity texts created: {len(entity_texts):,}")
print("\nExample entity texts:")
for entity in list(entity2id.keys())[:3]:
    text = entity_texts[entity]
    display_text = text[:100] + '...' if len(text) > 100 else text
    print(f"  {entity}")
    print(f"    -> {display_text}")

In [ ]:
def load_relation_aliases(file_path):
    """
    Load relation names and aliases from wikidata5m_relation.txt.
    Format: relation_id<TAB>name<TAB>alias1<TAB>alias2...
    """
    relation_info = {}
    print(f"Loading relation aliases from {file_path}...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                relation_id = parts[0]
                # Use primary name and first few aliases
                names = parts[1:4]  # Take up to 3 names/aliases
                combined_name = ', '.join(names)
                relation_info[relation_id] = combined_name
            elif len(parts) == 1:
                relation_id = parts[0]
                relation_info[relation_id] = relation_id
    return relation_info

# Load relation aliases
relation_aliases = load_relation_aliases(RAW_DATA_DIR / 'wikidata5m_relation.txt')
print(f"Loaded {len(relation_aliases):,} relation descriptions")

# Create relation texts
relation_texts = {}
for relation in relation2id.keys():
    if relation.endswith('_inv'):
        # Inverse relation
        original = relation[:-4]
        if original in relation_aliases:
            text = 'inverse of ' + relation_aliases[original]
        else:
            text = 'inverse of ' + original
    else:
        # Original relation
        if relation in relation_aliases:
            text = relation_aliases[relation]
        else:
            text = relation
    
    relation_texts[relation] = text

print(f"\nRelation texts created: {len(relation_texts):,}")
print("\nExample relation texts:")
for relation in list(relation2id.keys())[:6]:
    print(f"  {relation}")
    print(f"    -> {relation_texts[relation]}")

## 5. Generate Text Embeddings

Use Sentence-BERT to encode all entities and relations into dense vectors.

**Note**: With ~4.8M entities, this will take significant time. Consider saving checkpoints.

In [ ]:
# Install sentence-transformers if needed
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers...")
    !pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer

import torch

# Load model
print(f"Loading embedding model: {EMBEDDING_MODEL}")
encoder = SentenceTransformer(EMBEDDING_MODEL)
encoder = encoder.to(DEVICE)

print(f"Model loaded on {DEVICE}")
print(f"Embedding dimension: {encoder.get_sentence_embedding_dimension()}")

In [ ]:
# Encode entities (in batches to avoid memory issues)
print("Encoding entities...")
print(f"⚠️  Processing {len(entity2id):,} entities - this may take a while")

entity_list = sorted(entity2id.keys(), key=lambda x: entity2id[x])
entity_text_list = [entity_texts[e] for e in entity_list]

# Encode in batches
entity_embeddings = encoder.encode(
    entity_text_list,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    device=DEVICE
)

print(f"\nEntity embeddings shape: {entity_embeddings.shape}")
print(f"Memory size: {entity_embeddings.nbytes / (1024**2):.2f} MB")

# Save intermediate result to avoid re-computation
np.save(OUTPUT_DIR / 'entity_embeddings_temp.npy', entity_embeddings)
print("✓ Entity embeddings saved (temporary)")

In [ ]:
# Encode relations
print("Encoding relations...")
relation_list = sorted(relation2id.keys(), key=lambda x: relation2id[x])
relation_text_list = [relation_texts[r] for r in relation_list]

relation_embeddings = encoder.encode(
    relation_text_list,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    device=DEVICE
)

print(f"\nRelation embeddings shape: {relation_embeddings.shape}")
print(f"Memory size: {relation_embeddings.nbytes / (1024**2):.2f} MB")

## 6. Convert Triples to Tensor Format

In [ ]:
def convert_triples_to_ids(triples, entity2id, relation2id, add_inverse=True):
    """
    Convert triples from strings to IDs.
    Optionally add inverse triples.
    """
    id_triples = []
    
    print(f"Converting {len(triples):,} triples to IDs...")
    for h, r, t in tqdm(triples, desc="Converting triples"):
        if h in entity2id and r in relation2id and t in entity2id:
            h_id = entity2id[h]
            r_id = relation2id[r]
            t_id = entity2id[t]
            id_triples.append((h_id, r_id, t_id))
            
            # Add inverse triple
            if add_inverse:
                r_inv = r + '_inv'
                if r_inv in relation2id:
                    r_inv_id = relation2id[r_inv]
                    id_triples.append((t_id, r_inv_id, h_id))
    
    return np.array(id_triples, dtype=np.int32)

# Convert all splits
train_ids = convert_triples_to_ids(train_triples, entity2id, relation2id, CREATE_INVERSE_RELATIONS)
valid_ids = convert_triples_to_ids(valid_triples, entity2id, relation2id, CREATE_INVERSE_RELATIONS)
test_ids = convert_triples_to_ids(test_triples, entity2id, relation2id, CREATE_INVERSE_RELATIONS)

print(f"\nTraining triples (with inverses): {len(train_ids):,}")
print(f"Validation triples (with inverses): {len(valid_ids):,}")
print(f"Test triples (with inverses): {len(test_ids):,}")
print(f"\nTotal processed triples: {len(train_ids) + len(valid_ids) + len(test_ids):,}")

if CREATE_INVERSE_RELATIONS:
    original_total = len(train_triples) + len(valid_triples) + len(test_triples)
    processed_total = len(train_ids) + len(valid_ids) + len(test_ids)
    print(f"Data augmentation: {original_total:,} -> {processed_total:,} ({processed_total/original_total:.1f}x)")

## 7. Create Ground Truth Dictionary

For filtered evaluation: Store all valid (h, r, t) triples to avoid penalizing correct predictions.

In [ ]:
def create_ground_truth_dict(train_ids, valid_ids, test_ids):
    """
    Create a dictionary mapping (h, r) -> set of valid tails.
    Used for filtered evaluation.
    """
    ground_truth = defaultdict(set)
    
    all_triples = np.concatenate([train_ids, valid_ids, test_ids], axis=0)
    
    print(f"Creating ground truth dictionary from {len(all_triples):,} triples...")
    for h, r, t in tqdm(all_triples, desc="Building ground truth"):
        ground_truth[(int(h), int(r))].add(int(t))
    
    # Convert to regular dict with lists
    ground_truth = {k: list(v) for k, v in ground_truth.items()}
    
    return ground_truth

ground_truth = create_ground_truth_dict(train_ids, valid_ids, test_ids)

print(f"\nGround truth entries: {len(ground_truth):,}")
print(f"Average tails per (h, r): {np.mean([len(v) for v in ground_truth.values()]):.2f}")

# Save ground truth
# Convert keys from tuples to strings for JSON
print("\nSaving ground truth dictionary...")
ground_truth_json = {f"{h},{r}": tails for (h, r), tails in ground_truth.items()}
with open(OUTPUT_DIR / 'ground_truth.json', 'w') as f:
    json.dump(ground_truth_json, f)

print("✓ Ground truth saved")

## 8. Save All Processed Data

In [ ]:
import torch

print("Saving processed data...")

# Save embeddings
torch.save(torch.from_numpy(entity_embeddings), OUTPUT_DIR / 'entity_embeddings.pt')
torch.save(torch.from_numpy(relation_embeddings), OUTPUT_DIR / 'relation_embeddings.pt')

# Save triples
torch.save(torch.from_numpy(train_ids), OUTPUT_DIR / 'train_triples.pt')
torch.save(torch.from_numpy(valid_ids), OUTPUT_DIR / 'valid_triples.pt')
torch.save(torch.from_numpy(test_ids), OUTPUT_DIR / 'test_triples.pt')

# Calculate average neighbors per entity (for top-k recommendation)
entity_neighbors = defaultdict(set)
for h, r, t in train_ids:
    entity_neighbors[h].add((t, r))
    entity_neighbors[t].add((h, r))
avg_neighbors = np.mean([len(neighbors) for neighbors in entity_neighbors.values()]) if entity_neighbors else 0

# Save metadata
metadata = {
    'num_entities': len(entity2id),
    'num_relations': len(relation2id),
    'num_original_relations': len([r for r in relation2id if not r.endswith('_inv')]),
    'embedding_dim': entity_embeddings.shape[1],
    'embedding_model': EMBEDDING_MODEL,
    'has_inverse_relations': CREATE_INVERSE_RELATIONS,
    'train_size': len(train_ids),
    'valid_size': len(valid_ids),
    'test_size': len(test_ids),
    'avg_neighbors_per_entity': float(avg_neighbors),
    'dataset': 'Wikidata5M'
}

with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Clean up temporary file
temp_file = OUTPUT_DIR / 'entity_embeddings_temp.npy'
if temp_file.exists():
    temp_file.unlink()

print("="*70)
print("DATA PREPARATION COMPLETE")
print("="*70)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nFiles saved:")
print(f"  • entity_embeddings.pt ({entity_embeddings.shape})")
print(f"  • relation_embeddings.pt ({relation_embeddings.shape})")
print(f"  • train_triples.pt ({len(train_ids):,} triples)")
print(f"  • valid_triples.pt ({len(valid_ids):,} triples)")
print(f"  • test_triples.pt ({len(test_ids):,} triples)")
print(f"  • entity2id.json, relation2id.json")
print(f"  • ground_truth.json")
print(f"  • metadata.json")
print(f"\nDataset statistics:")
for key, value in metadata.items():
    print(f"  • {key}: {value}")
print(f"\n✓ Ready for context embedding generation!")

## 9. Generate Context Embeddings

**CRITICAL - NO DATA LEAKAGE**: Context is computed from **training triples ONLY** and used for all splits.

### Top-K Recommendation
- Average neighbors per entity: See metadata above
- **If avg_neighbors > 50**: Use `--top_k 20` to reduce over-smoothing
- **If avg_neighbors < 50**: NO top-k filtering (preserve all neighbors)

Wikidata5M is typically dense, so top-k filtering is recommended.

In [ ]:
required_files = ['generate_context_embeddings.py']

print("="*70)
print("Cloning GitHub repository...")
print("="*70)

# Clone your GitHub repo
GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
BRANCH = "context-aware-gwm-rnn"

!git clone {GITHUB_REPO} /kaggle/working/gwm
%cd /kaggle/working/gwm
!git checkout {BRANCH}
!git pull
%cd ../

# Copy files from repo to working directory
repo_path = "/kaggle/working/gwm/gwm-rnn/relation-prediction"

print(f"\nCopying files from {repo_path}...")
for file in required_files:
    !cp {repo_path}/{file} /kaggle/working/
    print(f"✓ Copied {file}")

# Verify files exist
import os
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")

In [ ]:
!python generate_context_embeddings.py \
    --data_dir /kaggle/working/dataset/ \
    --aggregation mean \
    -- top_k 20

## 10. Data Statistics and Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Relation frequency analysis
relation_counts = defaultdict(int)
for h, r, t in train_ids:
    relation_counts[r] += 1

# Plot relation distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Top 20 relations
top_relations = sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)[:20]
rel_names = [id2relation[r] for r, _ in top_relations]
rel_counts = [c for _, c in top_relations]

axes[0].barh(range(len(rel_names)), rel_counts)
axes[0].set_yticks(range(len(rel_names)))
axes[0].set_yticklabels([r[:40] + '...' if len(r) > 40 else r for r in rel_names], fontsize=8)
axes[0].set_xlabel('Number of Triples')
axes[0].set_title('Top 20 Most Frequent Relations')
axes[0].invert_yaxis()

# 2. Distribution of relation frequencies
counts = list(relation_counts.values())
axes[1].hist(counts, bins=50, edgecolor='black')
axes[1].set_xlabel('Number of Triples')
axes[1].set_ylabel('Number of Relations')
axes[1].set_title('Distribution of Relation Frequencies')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'relation_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Most frequent relation: {id2relation[top_relations[0][0]]} ({top_relations[0][1]:,} triples)")
print(f"Least frequent relation: {id2relation[top_relations[-1][0]]} ({top_relations[-1][1]:,} triples)")
print(f"Average triples per relation: {np.mean(counts):.1f}")
print(f"Median triples per relation: {np.median(counts):.1f}")

In [ ]:
# Entity degree distribution
entity_degrees = defaultdict(int)
for h, r, t in train_ids:
    entity_degrees[h] += 1
    entity_degrees[t] += 1

degrees = list(entity_degrees.values())

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.hist(degrees, bins=100, edgecolor='black')
ax.set_xlabel('Entity Degree (Number of Connections)')
ax.set_ylabel('Number of Entities')
ax.set_title('Distribution of Entity Degrees')
ax.set_yscale('log')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'entity_degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nEntity Degree Statistics:")
print(f"  Mean degree: {np.mean(degrees):.2f}")
print(f"  Median degree: {np.median(degrees):.2f}")
print(f"  Max degree: {np.max(degrees):,}")
print(f"  Min degree: {np.min(degrees):,}")
print(f"\n✓ Statistics saved to {OUTPUT_DIR}")